## Variational Inference — Régression Logistique Bayésienne


- Dataset  : Breast Cancer (sklearn)
- Modèle   : Régression logistique bayésienne
- Algo     : SVI (Stochastic Variational Inference) via Pyro

In [7]:
!pip install pyro-ppl scikit-learn

In [8]:

import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    precision_score, classification_report, confusion_matrix
)
from pyro.infer.autoguide import AutoNormal

In [9]:
# DATASET

data = load_breast_cancer()
X, y = data.data, data.target

X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

n_features = X_train.shape[1]
print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]} | Features : {n_features}\n")

Train : 455 | Test : 114 | Features : 30



In [10]:
#  MODÈLE 

def model(X, y):
    w = pyro.sample("w", dist.Normal(
        torch.zeros(n_features), torch.ones(n_features)
    ).to_event(1))
    b = pyro.sample("b", dist.Normal(0., 1.))
    with pyro.plate("data", len(X)):
        pyro.sample("obs", dist.Bernoulli(logits=X @ w + b), obs=y)

# GUIDE
guide = AutoNormal(model)

In [11]:
# ENTRAÎNEMENT 
pyro.clear_param_store()
svi = SVI(model, guide, Adam({"lr": 0.01}), loss=Trace_ELBO())

N_STEPS = 2000
print("Entraînement VI...")
for step in range(N_STEPS):
    loss = svi.step(X_train, y_train)
    if step % 400 == 0:
        print(f"  Step {step:4d} / {N_STEPS} | ELBO loss : {loss:.1f}")# ENTRAÎNEMENT 
pyro.clear_param_store()
svi = SVI(model, guide, Adam({"lr": 0.01}), loss=Trace_ELBO())

N_STEPS = 2000
print("Entraînement VI...")
for step in range(N_STEPS):
    loss = svi.step(X_train, y_train)
    if step % 400 == 0:
        print(f"  Step {step:4d} / {N_STEPS} | ELBO loss : {loss:.1f}")

Entraînement VI...
  Step    0 / 2000 | ELBO loss : 413.7
  Step  400 / 2000 | ELBO loss : 71.1
  Step  800 / 2000 | ELBO loss : 60.6
  Step 1200 / 2000 | ELBO loss : 54.4
  Step 1600 / 2000 | ELBO loss : 56.7
Entraînement VI...
  Step    0 / 2000 | ELBO loss : 61.8
  Step  400 / 2000 | ELBO loss : 67.6
  Step  800 / 2000 | ELBO loss : 58.0
  Step 1200 / 2000 | ELBO loss : 57.1
  Step 1600 / 2000 | ELBO loss : 55.5


In [ ]:
from pyro.infer import Predictive

predictive = Predictive(model, guide=guide, num_samples=200)
samples = predictive(X_test, None)  # None car obs pas connu au test
preds       = samples["obs"].float()
mean_prob   = preds.mean(0)
uncertainty = preds.std(0)
y_pred      = (mean_prob > 0.5).float()

y_true_np = y_test.numpy()
y_pred_np = y_pred.numpy()

In [14]:

# ÉVALUATION
print("=" * 50)
print(f"Accuracy           : {accuracy_score(y_true_np, y_pred_np):.4f}")
print(f"Precision (macro)  : {precision_score(y_true_np, y_pred_np, average='macro'):.4f}")
print(f"Recall    (macro)  : {recall_score(y_true_np, y_pred_np, average='macro'):.4f}")
print(f"F1-score  (macro)  : {f1_score(y_true_np, y_pred_np, average='macro'):.4f}")
print(f"Incertitude moy.   : {uncertainty.mean().item():.4f}")
print("=" * 50)

print("\nClassification Report :")
print(classification_report(y_true_np, y_pred_np,
      target_names=["Maligne (0)", "Bénigne (1)"]))

print("Matrice de confusion :")
print(confusion_matrix(y_true_np, y_pred_np))


Accuracy           : 0.9649
Precision (macro)  : 0.9588
Recall    (macro)  : 0.9673
F1-score  (macro)  : 0.9627
Incertitude moy.   : 0.1051

Classification Report :
              precision    recall  f1-score   support

 Maligne (0)       0.93      0.98      0.95        42
 Bénigne (1)       0.99      0.96      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.97      0.96       114
weighted avg       0.97      0.96      0.97       114

Matrice de confusion :
[[41  1]
 [ 3 69]]
